# DPS crystal prior — Colab runner

This notebook contains **no logic**. It clones the repo, installs deps, mounts Drive
and calls the same CLI scripts you run locally. All code lives in git so the local
CPU test loop stays meaningful — edit there, push, re-run here.

**Order matters.** Sections 5–7 are a ~1h cheap read that surfaces problems before
you commit many GPU-hours to sections 8–9. Don't skip ahead: a wiring or convention
error found after a full pretrain costs a day.

**Runtime → Change runtime type → GPU** before starting.


In [ ]:
!nvidia-smi
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 1. Repo


In [ ]:
import os, glob
REPO   = '/content/diffusion-posterior-sampling'
URL    = 'https://github.com/dawidratynski/diffusion-posterior-sampling.git'
BRANCH = 'scratchpad'   # <- the branch holding this work; change when merged to main

if os.path.exists(REPO):
    !cd {REPO} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull
else:
    !git clone --branch {BRANCH} {URL} {REPO}
%cd {REPO}

# Confirm you actually have the new code, not an older branch.
!git rev-parse --abbrev-ref HEAD && git log --oneline -1
assert os.path.exists('scripts/train_diffusion.py'), \
    'Wrong branch: scripts/train_diffusion.py is missing.'


## 2. Dependencies

`torch` is deliberately **not** a project dependency (it lives in the `cpu` group of
`pyproject.toml`, used only for local CPU dev), so this installs the project's other
deps and leaves Colab's CUDA torch untouched.


In [ ]:
!pip install -q -e .

# Fail loudly rather than limping on with missing deps: a failed editable
# install is easy to miss in Colab's output and only bites much later.
import importlib
for m in ['numpy', 'scipy', 'skimage', 'PIL', 'yaml', 'matplotlib', 'tqdm']:
    importlib.import_module(m)
print('dependencies OK')


## 3. Data (local disk) and checkpoints (Drive)

**Checkpoints go to Drive** — Colab disconnects and `/content` is lost.
**Data goes to local disk** — training reads thousands of small PNGs per epoch
and mounted-Drive I/O would starve the GPU.

The archive is located by searching for the directory containing `real/train`,
so it works whether or not the zip has a top-level folder inside it.

Set `DRIVE` and `ZIP` to match your Drive layout.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE  = '/content/drive/MyDrive/Non-movables'   # <- your Drive folder
ZIP    = f'{DRIVE}/dataset.zip'                  # <- your archive
MODELS = f'{DRIVE}/models'      # checkpoints on Drive: sessions die
os.makedirs(MODELS, exist_ok=True)

def latest_ckpt(d):
    """Newest model_ema_*.pt in d.

    Training cells save every --save_every steps, so you can interrupt one
    at any point and still have a usable checkpoint; this picks it up.
    """
    c = sorted(glob.glob(f'{d}/model_ema_*.pt'))
    if not c:
        raise FileNotFoundError(
            f'No model_ema_*.pt in {d}. Let the training cell reach at least one --save_every interval before running this.')
    return c[-1]

# Data goes on LOCAL disk, not Drive: training reads ~11.6k small PNGs per
# epoch and mounted-Drive I/O on many small files starves the GPU.
EXTRACT = '/content/data_raw'
if not os.path.exists(EXTRACT):
    !mkdir -p {EXTRACT} && unzip -q '{ZIP}' -d {EXTRACT}

# Archives may or may not carry a top-level folder, so find the directory
# that actually contains real/train rather than assuming the depth.
matches = sorted(glob.glob(f'{EXTRACT}/**/real/train', recursive=True))
assert matches, f'no */real/train found under {EXTRACT}; check the archive'
DATA = os.path.dirname(os.path.dirname(matches[0]))
print('DATA   =', DATA)
print('MODELS =', MODELS)

!ls {DATA}/real/train | wc -l && ls {DATA}/synth/train | wc -l


## 4. Split hygiene — **run this, do not just dry-run it**

Several crops come from each source photo, so the split must be at *source*
level; splitting on filenames leaks siblings across train/val and makes the
memorisation check meaningless.

**The archive on Drive predates the fix**, so the copy you just unpacked still
leaks. Run the first cell to confirm, then the second to repair it. Idempotent
— safe to re-run, and it skips domains that are already clean.


In [ ]:
!python scripts/fix_split.py {DATA}/real {DATA}/synth --dry_run


In [ ]:
# Apply it (expect ~4.6k real files to move; synth is already clean).
!python scripts/fix_split.py {DATA}/real {DATA}/synth


## 5. Environment smoke (~1 min)

The full train → generate → evaluate path at tiny scale. Proves the environment,
data layout and configs work together before anything expensive. Takes ~6 min on
CPU locally; much less here.

It also asserts the exported EMA checkpoint beats chance — guarding a bug where the
training log looked healthy while the saved model was still random initialisation.


In [ ]:
!PY=python ./scripts/smoke_e2e.sh {DATA} /content/smoke


## 6. Short real-only run (~1h) — the cheap read

A first prior at the **real** 160px resolution, trained only long enough to produce
meaningful output. This is deliberately *not* the final model: it exists to answer
"does DPS through this operator do anything sensible" in an hour rather than a day.

Checkpoints land every 1000 steps, so **you can interrupt this cell whenever you
like** and still use the latest one. Watch the `it/s` in the log to judge how far to
let it go.


In [ ]:
!python scripts/train_diffusion.py \
    --model_config configs/crystal_model_config.yaml \
    --data_root {DATA}/real/train \
    --out_dir {MODELS}/quick_real \
    --batch_size 16 --lr 1e-4 --train_steps 8000 \
    --save_every 1000 --amp --resume


## 7. Generate + evaluate from the quick run

`--mode validate` derives `y = A(real image)`, which is self-consistent for any
operator including the untrained `spectral` stand-in. (`--mode generate`, the actual
product, needs the trained UVCGAN generator — the stand-in's output distribution is
not the true synth domain, so the data-consistency term would be unsatisfiable.)


In [ ]:
CKPT = latest_ckpt(f'{MODELS}/quick_real'); print('using', CKPT)
!sed 's|^model_path:.*|model_path: {CKPT}|' configs/crystal_model_config.yaml > /content/model_cfg.yaml

# DPS costs a UNet forward AND backward per step, so 1000 steps x many
# samples runs to hours. Use 100 steps for the cheap read; it is enough to
# judge whether the output is sane, and section 11 uses the full schedule.
!sed 's|^timestep_respacing:.*|timestep_respacing: 100|' \
    configs/crystal_diffusion_config.yaml > /content/diffusion_fast.yaml

# Time ONE sample before committing to a batch.
import time; t0 = time.time()
!python scripts/generate_augmented.py \
    --model_config /content/model_cfg.yaml \
    --diffusion_config /content/diffusion_fast.yaml \
    --task_config configs/crystal_cyclegan_config.yaml \
    --mode validate --reference_root {DATA}/real/val \
    --out_dir /content/results/timing --method dps \
    --samples_per_input 1 --limit 1
per_sample = time.time() - t0
print(f'\n~{per_sample:.0f}s per 100-step sample.')
print(f'32 refs x 4 variants  = {128*per_sample/60:.0f} min at 100 steps')
print(f'full 1000-step sample = {per_sample*10:.0f}s each')


In [ ]:
# Adjust --limit from the timing above if needed.
!python scripts/generate_augmented.py \
    --model_config /content/model_cfg.yaml \
    --diffusion_config /content/diffusion_fast.yaml \
    --task_config configs/crystal_cyclegan_config.yaml \
    --mode validate --reference_root {DATA}/real/val \
    --out_dir /content/results/quick --method dps \
    --samples_per_input 4 --limit 32


In [ ]:
!python scripts/evaluate.py \
    --result_dir /content/results/quick \
    --real_root {DATA}/real/val \
    --real_train_root {DATA}/real/train \
    --out_csv /content/results/quick_per_image.csv


### Look at the images, not only the table

The metrics cannot tell you whether the output is plausible to a human eye. Compare
a generated image against the reference it was derived from.


In [ ]:
import matplotlib.pyplot as plt, csv
rows = list(csv.DictReader(open('/content/results/quick/manifest.csv')))[:4]
fig, ax = plt.subplots(2, len(rows), figsize=(3*len(rows), 6))
for i, r in enumerate(rows):
    ax[0, i].imshow(plt.imread(r['source_image'])); ax[0, i].set_title('reference', fontsize=9)
    ax[1, i].imshow(plt.imread(f"/content/results/quick/generated/{r['output_image']}"))
    ax[1, i].set_title('DPS', fontsize=9)
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.show()


## 8. `scale` sweep (~75 min)

DPS is very sensitive to the conditioning scale, and `0.3` is the FFHQ default,
not a tuned value for this task. Comparing an untuned DPS against UVCGAN would
understate DPS, so establish the range before the expensive runs.

**What to look for.** These pull in opposite directions:

- **label validity** (spacing / angle / signature) keeps improving as scale rises
  — the measurement is pinning the lattice down harder;
- **diversity** collapses once scale is too high, because every sample converges
  onto the same solution. That is the property DPS exists to provide, so winning
  label validity by killing it is a bad trade;
- **prominence** should land near the real value, not as high as possible.

Pick the largest scale that has not yet damaged diversity. The cell skips scales
already computed, so it resumes cleanly if the session drops.


In [ ]:
# 24 distinct source photos (subsets are strided, not truncated), 2 variants
# each. 2.0 is included because the interesting transition sits between 1.0
# and 3.0: label validity keeps improving while diversity starts collapsing.
SCALES = [0.3, 1.0, 2.0, 3.0]
N_REFS, N_VARIANTS = 24, 2

# Results from before the strided-subset fix used only ~2 distinct photos and
# must not be reused. Leave True unless you are resuming a sweep that was
# already run with the current code.
DISCARD_OLD = True
if DISCARD_OLD:
    !rm -rf /content/results/scale_*

print(f'{len(SCALES)} scales x {N_REFS*N_VARIANTS} samples x 100 steps')
print(f'at ~23 s/sample that is ~{len(SCALES)*N_REFS*N_VARIANTS*23/60:.0f} min total')

for s in SCALES:
    out = f'/content/results/scale_{s}'
    # Skip completed scales so an interrupted sweep resumes rather than
    # restarting from the first one.
    if os.path.exists(f'{out}/manifest.csv'):
        print(f'scale {s}: already present, skipping')
        continue
    !python scripts/generate_augmented.py \
        --model_config /content/model_cfg.yaml \
        --diffusion_config /content/diffusion_fast.yaml \
        --task_config configs/crystal_cyclegan_config.yaml \
        --mode validate --reference_root {DATA}/real/val \
        --out_dir {out} --method dps \
        --scale {s} --samples_per_input {N_VARIANTS} --limit {N_REFS}

# Confirm the subsets really are spread across photos, not crops of one scene.
import csv, re
for s in SCALES:
    rows = list(csv.DictReader(open(f'/content/results/scale_{s}/manifest.csv')))
    photos = {re.sub(r'_sample_\d+\.png$', '', os.path.basename(r['source_image']))
              for r in rows}
    print(f'scale {s}: {len(rows)} images from {len(photos)} distinct photos')

dirs = ' '.join(f'--result_dir /content/results/scale_{s}' for s in SCALES)
!python scripts/evaluate.py {dirs} --real_root {DATA}/real/val


---
# Only continue once sections 5–7 look right.
Everything below is measured in GPU-hours.
---


## 9. Stage 1 — pretrain on SYNTH

Synth is effectively unlimited, so this stage carries no memorisation risk and
teaches lattice structure and low-level statistics. Keep `--resume` on: re-running
the cell after a disconnect continues rather than restarting.


In [ ]:
!python scripts/train_diffusion.py \
    --model_config configs/crystal_model_config.yaml \
    --data_root {DATA}/synth/train \
    --out_dir {MODELS}/synth_pretrain \
    --batch_size 16 --lr 1e-4 --train_steps 150000 \
    --save_every 5000 --amp --resume


## 10. Stage 2 — finetune on REAL

`--init_from` starts from the pretrained weights with a fresh optimiser and step
counter (unlike `--resume`, which continues a run). Real is the scarce side —
~2.9k source photos — so **stop on the memorisation metric, not on loss**.

Re-run section 11 against successive checkpoints and watch `NN dist to real train`:
when it falls toward the printed floor, the prior is starting to copy and further
training makes the augmentation worse, however good realism looks.


In [ ]:
PRETRAINED = latest_ckpt(f'{MODELS}/synth_pretrain'); print('init from', PRETRAINED)
!python scripts/train_diffusion.py \
    --model_config configs/crystal_model_config.yaml \
    --data_root {DATA}/real/train \
    --out_dir {MODELS}/real_finetune \
    --init_from {PRETRAINED} \
    --batch_size 16 --lr 2e-5 --train_steps 40000 \
    --save_every 2000 --amp --resume


## 11. Final generate + evaluate

**Budget this before launching.** DPS costs a UNet forward *and* backward per
step, so 200 references x 4 variants at the full 1000-step schedule is 800
samples — from ~6h to well over a day depending on the GPU. Use the per-sample
time printed in section 7 (x10 for the full schedule) to pick `--limit`. The
metrics stabilise well before 200 references; 50-100 is usually plenty.

Once UVCGAN weights exist: set `framework: uvcgan2` and `path:` in the task
config, switch to `--mode generate --synth_root`, and add a `--method uvcgan`
run as the baseline. Both write the same manifest format, so one `evaluate.py`
call compares them side by side.


In [ ]:
CKPT = latest_ckpt(f'{MODELS}/real_finetune'); print('using', CKPT)
!sed 's|^model_path:.*|model_path: {CKPT}|' configs/crystal_model_config.yaml > /content/model_cfg.yaml

!python scripts/generate_augmented.py \
    --model_config /content/model_cfg.yaml \
    --diffusion_config configs/crystal_diffusion_config.yaml \
    --task_config configs/crystal_cyclegan_config.yaml \
    --mode validate --reference_root {DATA}/real/val \
    --out_dir /content/results/final_dps --method dps \
    --samples_per_input 4 --limit 200

!python scripts/evaluate.py \
    --result_dir /content/results/final_dps \
    --real_root {DATA}/real/val \
    --real_train_root {DATA}/real/train \
    --out_csv /content/results/final_per_image.csv

!cp -r /content/results {DRIVE}/results_$(date +%Y%m%d_%H%M)


---
### If the session dies
Re-run sections 1–3, then the training cell — `--resume` picks up from the last
checkpoint on Drive. Nothing is lost beyond the last `--save_every` interval.

### What none of this proves
With the `spectral` stand-in you are testing DPS against an analytic caricature of
`G_{R→S}`. Good numbers mean the machinery is sound, **not** that the thesis claim
holds. That verdict needs the trained UVCGAN generator.
